In [ ]:
from pathlib import Path
from datetime import datetime, date
import re
import sys

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter
from openpyxl.utils.datetime import from_excel


# =====================================================
# ONLY CURRENT WORKING DIRECTORY
# =====================================================

BASE_DIR = Path.cwd()
OUTPUT_FILE = BASE_DIR / "PO_Items_Consolidated.xlsx"

print("Working directory:", BASE_DIR)


FINAL_HEADERS = [
    "PO",
    "Vendor Code",
    "ASIN",
    "External ID",
    "Model Number",
    "Title",
    "Availability",
    "Window Type",
    "Window Start",
    "Window End",
    "Expected Date",
    "Quantity Requested",
    "Quantity Accepted",
    "Quantity Received",
    "Quantity Outstanding",
    "Unit Cost",
    "Currency",
    "Total Cost",
    "Currency",
]


HEADER_MAP = {
    "PO": ["po"],
    "Vendor Code": ["vendor", "vendor code"],
    "ASIN": ["asin"],
    "External ID": ["external id", "externalid"],
    "Model Number": ["model number", "model no", "model numebr"],
    "Title": ["title", "item title", "product title"],
    "Availability": ["availability", "availibility"],
    "Window Type": ["window type"],
    "Window Start": ["window start", "window start date"],
    "Window End": ["window end", "window end date"],
    "Expected Date": ["expected date"],
    "Quantity Requested": ["quantity requested", "requested quantity"],
    "Quantity Accepted": ["accepted quantity", "quantity accepted"],
    "Quantity Received": ["quantity received", "received quantity"],
    "Quantity Outstanding": ["quantity outstanding", "outstanding quantity"],
    "Unit Cost": ["unit cost"],
    "Total Cost": ["total cost"],
}


CRITICAL_COLUMNS = list(HEADER_MAP.keys())


def norm(x):
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"[_\-]+", " ", x)
    x = re.sub(r"[^a-z0-9 ]+", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return str(x).strip()


def clean_number(x):
    if x is None or str(x).strip() == "":
        return None
    if isinstance(x, (int, float)):
        return x
    x = str(x).replace(",", "").replace("₹", "").replace("INR", "").strip()
    try:
        return float(x)
    except:
        return x


def clean_date(x):
    if x is None or str(x).strip() == "":
        return None

    if isinstance(x, datetime):
        return x.date()

    if isinstance(x, date):
        return x

    if isinstance(x, (int, float)):
        try:
            return from_excel(x).date()
        except:
            return x

    text = str(x).strip()

    for fmt in ("%d/%m/%Y", "%m/%d/%Y", "%Y-%m-%d", "%d-%m-%Y", "%m-%d-%Y"):
        try:
            return datetime.strptime(text, fmt).date()
        except:
            pass

    return text


def find_header_row(rows):
    expected = set()
    for names in HEADER_MAP.values():
        expected.update(norm(n) for n in names)

    best_index = None
    best_score = 0

    for i, row in enumerate(rows[:20]):
        score = sum(1 for cell in row if norm(cell) in expected)
        if score > best_score:
            best_score = score
            best_index = i

    if best_score < 8:
        return None

    return best_index


def get_mapping(header_row):
    normalized_headers = {}

    for idx, value in enumerate(header_row):
        key = norm(value)
        if key:
            normalized_headers[key] = idx

    mapping = {}

    for final_col, possible_names in HEADER_MAP.items():
        found = None
        for name in possible_names:
            key = norm(name)
            if key in normalized_headers:
                found = normalized_headers[key]
                break
        mapping[final_col] = found

    return mapping


# =====================================================
# READ ONLY FILES FROM CURRENT FOLDER
# =====================================================

input_files = sorted([
    f for f in BASE_DIR.glob("PurchaseOrderItems*.xlsx")
    if not f.name.startswith("~$")
    and f.name != OUTPUT_FILE.name
])

if not input_files:
    raise SystemExit("No PurchaseOrderItems*.xlsx files found in current working directory only.")

print("Files found:")
for f in input_files:
    print(" -", f.name)


master_rows = []
seen = set()
validation_rows = []


for file_path in input_files:
    print("Processing:", file_path.name)

    wb = load_workbook(file_path, data_only=True, read_only=True)

    if "PurchaseOrderItems" in wb.sheetnames:
        ws = wb["PurchaseOrderItems"]
    else:
        ws = wb[wb.sheetnames[0]]

    rows = list(ws.iter_rows(values_only=True))

    header_idx = find_header_row(rows)

    if header_idx is None:
        wb.close()
        raise SystemExit(f"Header row not found in {file_path.name}. Stopping safely.")

    header_row = rows[header_idx]
    mapping = get_mapping(header_row)

    missing = [c for c in CRITICAL_COLUMNS if mapping.get(c) is None]

    if missing:
        wb.close()
        raise SystemExit(
            f"Critical columns missing in {file_path.name}: {missing}\n"
            "Stopped safely. Output not created."
        )

    for source_row_no, row in enumerate(rows[header_idx + 1:], start=header_idx + 2):

        po = clean_text(row[mapping["PO"]])
        asin = clean_text(row[mapping["ASIN"]])
        vendor = clean_text(row[mapping["Vendor Code"]])

        if not po and not asin and not vendor:
            continue

        out = [
            po,
            vendor,
            clean_text(row[mapping["ASIN"]]),
            clean_text(row[mapping["External ID"]]),
            clean_text(row[mapping["Model Number"]]),
            clean_text(row[mapping["Title"]]),
            clean_text(row[mapping["Availability"]]),
            clean_text(row[mapping["Window Type"]]),
            clean_date(row[mapping["Window Start"]]),
            clean_date(row[mapping["Window End"]]),
            clean_date(row[mapping["Expected Date"]]),
            clean_number(row[mapping["Quantity Requested"]]),
            clean_number(row[mapping["Quantity Accepted"]]),
            clean_number(row[mapping["Quantity Received"]]),
            clean_number(row[mapping["Quantity Outstanding"]]),
            clean_number(row[mapping["Unit Cost"]]),
            "INR",
            clean_number(row[mapping["Total Cost"]]),
            "INR",
        ]

        key = tuple(str(x).strip() if x is not None else "" for x in out)

        if key in seen:
            continue

        seen.add(key)
        master_rows.append(out)

    wb.close()


if not master_rows:
    raise SystemExit("No valid rows found. Output not created.")


# =====================================================
# CREATE FINAL EXCEL
# =====================================================

wb_out = Workbook()
ws = wb_out.active
ws.title = "PO_Items_Master"

ws.append(FINAL_HEADERS)

for row in master_rows:
    ws.append(row)


# Header styling: subtle
header_fill = PatternFill("solid", fgColor="D9EAF7")
header_font = Font(bold=True, color="1F1F1F")
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.border = thin_border
    cell.alignment = Alignment(horizontal="center", vertical="center")

for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(vertical="top")


# Freeze and filter
ws.freeze_panes = "A2"

last_row = ws.max_row
last_col = ws.max_column
table_ref = f"A1:{get_column_letter(last_col)}{last_row}"

table = Table(displayName="PO_Items_Table", ref=table_ref)
style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
table.tableStyleInfo = style
ws.add_table(table)
ws.auto_filter.ref = table_ref


# Formats
date_cols = [9, 10, 11]
qty_cols = [12, 13, 14, 15]
cost_cols = [16, 18]

for col in date_cols:
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=col, max_col=col):
        row[0].number_format = "dd-mmm-yyyy"

for col in qty_cols:
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=col, max_col=col):
        row[0].number_format = "0"

for col in cost_cols:
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=col, max_col=col):
        row[0].number_format = "#,##0.00"


# Widths
widths = {
    "A": 16, "B": 14, "C": 14, "D": 16, "E": 18,
    "F": 45, "G": 14, "H": 14, "I": 14, "J": 14,
    "K": 14, "L": 18, "M": 18, "N": 18, "O": 20,
    "P": 14, "Q": 12, "R": 14, "S": 12,
}

for col, width in widths.items():
    ws.column_dimensions[col].width = width

for cell in ws["F"][1:]:
    cell.alignment = Alignment(wrap_text=True, vertical="top")


wb_out.save(OUTPUT_FILE)

print("Done.")
print("Final file created:", OUTPUT_FILE)
print("Files processed:", len(input_files))
print("Rows created:", len(master_rows))

Working directory: c:\Users\ayush\Desktop\Projects\inventory_cover
Files found:
 - PurchaseOrderItems (1).xlsx
 - PurchaseOrderItems (2).xlsx
 - PurchaseOrderItems (3).xlsx
 - PurchaseOrderItems.xlsx
Processing: PurchaseOrderItems (1).xlsx
Processing: PurchaseOrderItems (2).xlsx
Processing: PurchaseOrderItems (3).xlsx
Processing: PurchaseOrderItems.xlsx
Done.
Final file created: c:\Users\ayush\Desktop\Projects\inventory_cover\PO_Items_Consolidated.xlsx
Files processed: 4
Rows created: 2715


: 


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

